# Modified Gram-Schmidt: My Investigation into Numerical Stability

In this notebook, I'm exploring a classic problem in numerical linear algebra: why do algorithms that are mathematically equivalent on paper behave so differently in a computer? Specifically, I'm looking at the **Classical Gram-Schmidt (CGS)** and **Modified Gram-Schmidt (MGS)** processes to see how they handle the buildup of floating-point errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Setting up the path to my library code
sys.path.append(os.path.abspath(os.path.join('..')))

from src.linalg.orthonormalization import modified_gram_schmidt, classical_gram_schmidt
from src.linalg.special_matrices import hilbert_matrix

## 1. Visualizing the Geometric Process

Before I dive into numerical stability, I want to make sure I have the basic geometry right. I'll take two vectors that are clearly not orthogonal and watch how the Gram-Schmidt process "straightens" them out into an orthonormal basis ($q_1, q_2$).

In [ ]:
# I'll start with two simple non-orthogonal vectors
v1 = np.array([1, 0.5])
v2 = np.array([0.8, 1.2])
A = np.column_stack((v1, v2))

# Applying my MGS implementation
Q, R = modified_gram_schmidt(A)
q1, q2 = Q[:, 0], Q[:, 1]

# Plotting to verify my intuition
plt.figure(figsize=(8, 8))
origin = np.array([[0, 0], [0, 0]])

plt.quiver(*origin, A[0, :], A[1, :], color=['r', 'b'], scale=1, scale_units='xy', label=['v1', 'v2'])
plt.quiver(*origin, Q[0, :], Q[1, :], color=['darkred', 'darkblue'], scale=1, scale_units='xy', label=['q1', 'q2'], alpha=0.5)

plt.xlim(-0.5, 2)
plt.ylim(-0.5, 2)
plt.axhline(0, color='black', lw=1)
plt.axvline(0, color='black', lw=1)
plt.grid(True, linestyle='--')
plt.title(r"My Geometric Test: v1, v2 $\rightarrow$ q1, q2")
plt.legend()
plt.show()

print(f"Self-check: Dot product of q1 and q2 is {np.dot(q1, q2):.16f}")

## 2. Pushing the Limits: The Hilbert Matrix Test

Now I want to test the **ReCoDE standard of Numerical Stability**. I've learned that the **Hilbert matrix** is notoriously difficult to work with because its columns become almost linearly dependent very quickly. This is the perfect "stress test" for Gram-Schmidt.

I'm going to compare CGS and MGS on a $10 \times 10$ Hilbert matrix and measure the "loss of orthogonality" by checking how far $Q^T Q$ is from the Identity matrix $I$.

In [ ]:
n_size = 10
H = hilbert_matrix(n_size)

Q_cgs, _ = classical_gram_schmidt(H)
Q_mgs, _ = modified_gram_schmidt(H)

# I'll measure the error using the Frobenius norm ||I - Q^T Q||
err_cgs = np.linalg.norm(np.eye(n_size) - Q_cgs.T @ Q_cgs)
err_mgs = np.linalg.norm(np.eye(n_size) - Q_mgs.T @ Q_mgs)

print(f"CGS Orthogonality Error (Loss): {err_cgs:.2e}")
print(f"MGS Orthogonality Error (Loss): {err_mgs:.2e}")

# I'll use the same color scale for both so the comparison is fair
vmax = max(np.max(np.abs(np.eye(n_size) - Q_cgs.T @ Q_cgs)), 
           np.max(np.abs(np.eye(n_size) - Q_mgs.T @ Q_mgs)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

im1 = ax1.imshow(np.abs(np.eye(n_size) - Q_cgs.T @ Q_cgs), cmap='viridis', vmin=0, vmax=vmax)
ax1.set_title(f"CGS: Orthogonality Loss (Error: {err_cgs:.2e})")
fig.colorbar(im1, ax=ax1)

im2 = ax2.imshow(np.abs(np.eye(n_size) - Q_mgs.T @ Q_mgs), cmap='viridis', vmin=0, vmax=vmax)
ax2.set_title(f"MGS: Staying Stable (Error: {err_mgs:.2e})")
fig.colorbar(im2, ax=ax2)

plt.show()

## My Reflection: Why does MGS win?

The results from the heatmaps above are pretty striking. While CGS and MGS look identical in my linear algebra textbook, my computer clearly sees a difference. 

I've realized that the "danger" in CGS comes from projecting the *original* vectors onto the growing subspace. If those original vectors are nearly dependent (like in the Hilbert matrix), the rounding errors from each projection just pile up. 

MGS is much smarter—it orthogonalizes the *remaining* vectors against each new basis vector as soon as it's found. This effectively "cleans up" the errors as we go. It's a great example of how a small change in the order of operations can be the difference between a reliable algorithm and one that fails when things get tough.